# Fine-Tuning the Champion Model on Merged Datasets

This notebook loads the `merged_metadata.csv` (HAM10000 + ISIC 2019 + PAD-UFES-20) and fine-tunes the existing `champion_transformer.pt` model.

**Hardware Target:** Local execution on NVIDIA RTX 3050. 
**Optimizations:** Batch size is reduced to 16 to prevent Out of Memory (OOM) errors on 4GB/8GB VRAM.

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm

import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

from src.model import build_champion_model
from src.dataset import SafeDermDataset
from src.transforms import get_train_transforms, get_eval_transforms
from src.config import CHAMPION_MODEL_PATH

### 1. Load Dataset

In [ ]:
data_dir = Path("../data")
csv_path = data_dir / "merged_dataset.csv"

if not csv_path.exists():
    raise FileNotFoundError("Please run scripts/prepare_unified_dataset.py first to generate the merged dataset.")

df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} images for fine-tuning.")

# Create Dataset and DataLoader
train_transforms = get_train_transforms()
train_dataset = SafeDermDataset(df, img_dir=data_dir, transform=train_transforms)

# Optimized for RTX 3050 (Batch Size = 16 or 32 to prevent OOM)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

### 2. Load Existing Champion Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = build_champion_model(num_classes=7)

model_path = Path("../models/champion_transformer.pt")
if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=device))
    print("Successfully loaded previous champion_transformer.pt weights.")
else:
    print("Warning: champion_transformer.pt not found. Training from pretrained ImageNet weights.")

model = model.to(device)

### 3. Fine-Tuning Loop

In [ ]:
# We use a very small learning rate because we are fine-tuning an already trained model
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({"Loss": running_loss/total, "Acc": correct/total})

print("Finished Fine-Tuning!")

### 4. Save Fine-Tuned Model

In [ ]:
save_path = Path("../models/finetuned_champion_transformer.pt")
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

print("\n--- NEXT STEPS ---")
print("1. Delete or rename the old champion_transformer.pt")
print("2. Rename finetuned_champion_transformer.pt -> champion_transformer.pt")
print("3. Run 08_calibration_conformal.ipynb to generate new thresholds!")